In [ ]:
# Paths are resolved relative to the repository, or to $PUBO_DATA if set.
# The original research code used absolute paths on the authors' machine;
# only the root has been parameterised, the directory layout is unchanged.
import os
DATA_ROOT = os.environ.get('PUBO_DATA',
                           os.path.join(os.path.dirname(os.getcwd()), '..', 'data', 'legacy'))
EXTERNAL_ROOT = os.environ.get('PUBO_EXTERNAL', DATA_ROOT)  # data not shipped with the repo


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
#import tf
import torch
import time
import math
from IPython.display import clear_output
import os
import mloop.interfaces as mli
import mloop.controllers as mlc
import mloop.visualizations as mlv
import tensorflow as tf

# torch.compat.v1.enable_eager_execution()

In [ ]:
### Solver ###
class TGD_class():
    #Initial parameters
    def __init__(self, K, dt,zeta, sigma, D_coeff, gamma_init_coeff, rho_thr_coeff,eps_coeff, MAX_ITER, N, attempt_num, unity, zeros):
        self.K = K.type(torch.complex64) 
        self.dt = dt
        self.MAX_ITER = MAX_ITER
        self.unity = unity.to('cpu')
        self.zeros=zeros.to('cpu')
        #scale = torch.dtypes.cast(Get_Scale(K,N), dtype='complex64')
        self.eps_coeff = eps_coeff
        self.rho_thr_coeff = rho_thr_coeff
        self.gamma_init_coeff = gamma_init_coeff #0.2*scale
        self.N=N
        self.attempt_num=attempt_num
        self.pi = np.pi
        self.zeta=zeta
        self.sigma=sigma
        self.D_coeff=D_coeff
        self.scale = torch.max(torch.einsum('ijk->i', torch.abs(self.K)))
#     def Get_Scale(self):
#         isum=torch.dtypes.cast(torch.einsum('ijk->i', torch.abs(self.K)), dtype='float')
#         scale = torch.reduce_max(isum)
#         return scale
    
    def GD_Solver(self, phi_old, phi_new, gamma, eps, rho_thr):
        Density = torch.abs(phi_old)**2
#         print(Density.size(), unity.size(), self.zeros.size(), rho_thr.size())
        D = self.D_coeff*torch.maximum(self.unity-Density/rho_thr,self.zeros) 
        Noise = self.sigma * 1j * math.pi * torch.rand(self.N, self.attempt_num, device = torch.device('cpu'))
        Noise = torch.exp(Noise)               
        dphi = torch.multiply(gamma - Density , phi_old)
        #for y in range(N):
        
        
        
        
        dphi-=self.zeta/self.N*torch.sqrt(rho_thr)*torch.einsum('ijl,js,ls->is', self.K, phi_old, torch.conj(phi_old)) #torch.outer(phi_old,torch.conj(phi_old))
        phi_new = phi_old + self.dt * (dphi + torch.multiply(D,Noise))  
        gamma = gamma + self.dt * eps * (rho_thr * self.unity - Density)
        return phi_new, gamma
    
    
    
    def GD_eval(self):
        eps =  self.eps_coeff*self.scale
        rho_thr = self.rho_thr_coeff*self.scale
        gamma_init = self.gamma_init_coeff*self.scale #0.2*scale
        #Initialisation
        phi_1 = torch.zeros(self.N,self.attempt_num, dtype = torch.complex64, device = torch.device('cpu'))
        phi_2 = phi_1.clone()
        
        gamma = -gamma_init * torch.ones((self.N,self.attempt_num), device = torch.device('cpu'))
        
        for m in range(self.MAX_ITER):
            phi_2, gamma = self.GD_Solver(phi_1, phi_2, gamma,  eps, rho_thr)
            phi_1, gamma = self.GD_Solver(phi_2, phi_1, gamma,  eps, rho_thr)
        spins=torch.sign(torch.cos(torch.angle(phi_1)))    
        self.lastcuts= torch.einsum('ijk,is,js,ks->s',self.K.to(dtype=torch.float32),spins,spins,spins)
        #
        if torch.sum(torch.isnan(self.lastcuts).type(dtype= torch.float))!=0:
            self.lastcuts=1000*torch.ones(self.attempt_num)
        return self.lastcuts, spins, phi_1, gamma
    
    
    
    
    
    def GD_eval_cc(self, phi_1, gamma, K1,K):
        self.K=K1
        self.K_en=K
        eps =  self.eps_coeff*self.scale
        rho_thr = self.rho_thr_coeff*self.scale
        #gamma_init = self.gamma_init_coeff*self.scale #0.2*scale
        #Initialisation
        #phi_1 = torch.zeros((self.N,self.attempt_num), dtype='complex64')
        phi_2 = phi_1.clone()
        #gamma = -gamma_init * torch.ones((self.N,self.attempt_num), dtype='complex64')
        for m in range(self.MAX_ITER):
            phi_2, gamma = self.GD_Solver(phi_1, phi_2, gamma,  eps, rho_thr)
            phi_1, gamma = self.GD_Solver(phi_2, phi_1, gamma,  eps, rho_thr)
        spins=torch.sign(torch.cos(torch.angle(phi_1)))    
        self.lastcuts= torch.einsum('ijk,is,js,ks->s',self.K_en.type(dtype=torch.float32),spins,spins,spins)
        #
        if torch.sum(torch.isnan(self.lastcuts).type(dtype= torch.float))!=0:
            self.lastcuts=1000*torch.ones(self.attempt_num)
        return self.lastcuts, spins, phi_1, gamma
    

In [ ]:
attempt_num=1000
MAX_ITER=1000


for i in range(2, 3):
    
    psize = int(10 * (i + 1))

    for number in range(0, 10):

        print("-------------------------")
        print(f"psize: {psize}   number: {number}")
        
        foldername = str(psize) + '_' + str(number)
        K = torch.tensor(np.load(DATA_ROOT+'/Sparse random/K_sparse_random_' + foldername + '_.npy'), dtype=torch.float, device=torch.device('cpu'))

        try:
            params = np.load(DATA_ROOT+"/Benchmark_each_final/sparse/tgd/" + f"{psize}_0" + 'M_params.npy',)       
        except: 
            print("M_parametrs нет")
            try:
                params = np.load(DATA_ROOT+"/Benchmark_each_final/sparse/tgd/" + f"{psize}_0" + '_params_tuned.npy',)
                
            except:
                print("_parametrs нет")
                params = [1, 1, 1, 1, 1, 1, 1]
        
        N = K.size(0)
        params = np.array([9.47987789e+00, 5.00018675e-03, 3.23425890e+00, 1.08659004e-02,
                           2.37850147e-01, 7.55611353e-03, 1.97464241e+01])

        unity=torch.ones((N,attempt_num),device = torch.device('cpu'))
        zeros=torch.zeros((N,attempt_num), device = torch.device('cpu'))
        sigma, dt, zeta,  eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff = np.array(params)

        percentile = 50


        TGD = TGD_class(K, dt, zeta, sigma, D_coeff, gamma_init_coeff, rho_thr_coeff, eps_coeff,MAX_ITER, N, attempt_num, unity, zeros)
        
        H_opt, spins_opt, phi, gamma=TGD.GD_eval()
        
        print("минимум до mloop", H_opt.min())
        print("среднее до mloop", np.mean(np.array(TGD.lastcuts.cpu())))
        np.save('GD_each/' + foldername + 'M0_spins.npy', spins_opt.cpu().numpy())
        np.save('GD_each/' + foldername + 'M0_energy.npy', H_opt.cpu().numpy())
           
        score_0 = H_opt.min() * 10 + np.mean(np.array(TGD.lastcuts.cpu()))
        
        

#         # MLOOP оптимизация
        
#         interface = TGDInterface(TGD,attempt_num,percentile)

#         first_params = np.array([np.log10(TGD.sigma), np.log10(TGD.dt), np.log10(TGD.zeta), np.log10(TGD.eps_coeff), np.log10(TGD.rho_thr_coeff), np.log10(TGD.gamma_init_coeff), np.log10(TGD.D_coeff)])
        
#         min_boundary = np.array([0,-4,-3,-4,-4,-4, 0])
#         max_boundary = np.array([3, 0, 2, 2, 2, 2, 2])
#         first_params = np.minimum(np.maximum(first_params, min_boundary), max_boundary)
        
#         num_params = 7
#         config = dict(controller_type='neural_net', 
#                       training_type = 'random',
#                       first_params = first_params,
#                       min_boundary = min_boundary, 
#                       max_boundary = max_boundary,
#                       num_params = num_params,
#                       trust_region = max_boundary - min_boundary,
#                       max_num_runs = 100 + psize * 5,
#                       num_training_runs = 2 * num_params,
#                       log_level = 0, 
#                       no_delay = False, 
#                       max_num_runs_without_better_params = 100 + psize * 3, 
#                       population_size=7, 
#                       mutation_scale = (0.0,0.2), 
#                       cross_over_probability = 0.7, 
#                       default_bad_cost = 1.05, 
#                       default_bad_uncertainty = 0.01,
#                       cost_has_noise = True)

#         controller = mlc.create_controller(interface, **config)
#         controller.optimize()
#         clear_output(True)
#      #   mlv.show_all_default_visualizations(controller)



#         sigma = 10**controller.best_params[0]
#         dt = 10**controller.best_params[1]
#         zeta = 10**controller.best_params[2]
#         eps_coeff = 10**controller.best_params[3]
#         rho_thr_coeff = 10**controller.best_params[4]
#         gamma_init_coeff = 10**controller.best_params[5]
#         D_coeff = 10**controller.best_params[6]   



#         params = [sigma, dt, zeta,  eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff]
#         unity = torch.ones((N,attempt_num), device = torch.device('cuda'))
#         zeros = torch.zeros((N,attempt_num), device = torch.device('cuda'))
#         TGD = TGD_class(K,dt, zeta, sigma, D_coeff, gamma_init_coeff, rho_thr_coeff,eps_coeff, MAX_ITER, N, attempt_num, unity, zeros)
#         H_opt, spins_opt, phi, gamma = TGD.GD_eval()
#         score_1 = H_opt.min() * 10 + np.mean(np.array(TGD.lastcuts.cpu()))
#         if number == 0:
            
#             print("минимум после mloop", H_opt.min())
#             print("среднее после mloop", np.mean(np.array(TGD.lastcuts.cpu())))
            
# #             if score_1 < score_0:
# #                 print("!!! замена начальных параметров")
# #                 np.save(DATA_ROOT+"/Benchmark_each_final/sparse/tgd/" + f"{psize}_0" + 'M_params.npy', params)
            
np.save('GD_each/' + foldername + 'M_spins.npy', spins_opt.cpu().numpy())
np.save('GD_each/' + foldername + 'M_energy.npy', H_opt.cpu().numpy())
np.save('GD_each/' + foldername + 'M_params.npy', params)
            
#         elif number > 0:
#             print("минимум после mloop", H_opt.min())
#             print("среднее после mloop", np.mean(np.array(TGD.lastcuts.cpu())))
#             print(score_1 - score_0)
np.save('GD_each/' + foldername + 'M_spins.npy', spins_opt.cpu().numpy())
np.save('GD_each/' + foldername + 'M_energy.npy', H_opt.cpu().numpy())
np.save('GD_each/' + foldername + 'M_params.npy', params)

        
